In [2]:
# FILE: 08_Voice_Test_Direct.ipynb

import os
import torch
import torchaudio
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

# 1. SETUP PATHS
model_dir = os.path.join(os.getenv("LOCALAPPDATA"), "tts", "tts_models--multilingual--multi-dataset--xtts_v2")
checkpoint_path = os.path.join(model_dir, "model.pth")
config_path = os.path.join(model_dir, "config.json")
vocab_path = os.path.join(model_dir, "vocab.json")
speaker_wav = "my_voice.wav"

print(f"🔹 Target Folder: {model_dir}")

# 2. SECURITY CHECK (Prevent the 'Ghost File' issue)
if os.path.exists(checkpoint_path):
    size_gb = os.path.getsize(checkpoint_path) / (1024**3)
    print(f"📦 Found 'model.pth': {size_gb:.2f} GB")
    if size_gb < 1.0:
        raise ValueError("❌ STOP! The file is too small (it got overwritten). Please copy the 1.8GB file again.")
else:
    raise FileNotFoundError(f"❌ Could not find model.pth in {model_dir}")

# 3. DIRECT LOAD (Bypassing the Manager)
print("⚙️ Loading Configuration...")
config = XttsConfig()
config.load_json(config_path)

print("🧠 Initializing Model...")
model = Xtts.init_from_config(config)

print("🏋️ Loading Weights (This may take 10s)...")
model.load_checkpoint(config, checkpoint_path, vocab_path=vocab_path, use_deepspeed=False)
model.cuda()

print("✅ Model Loaded! (The Manager did not touch your file)")

# 4. GENERATE AUDIO
print("🎙️ Cloning your voice...")

# Get the "Fingerprint" of your voice
gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(audio_path=[speaker_wav])

# Speak
out = model.inference(
    text="This is the final victory. I am speaking with your voice, and the system is ready.",
    language="en",
    gpt_cond_latent=gpt_cond_latent,
    speaker_embedding=speaker_embedding,
    temperature=0.7, # Creativity
)

# Save
output_file = "output_direct.wav"
torchaudio.save(output_file, torch.tensor(out["wav"]).unsqueeze(0), 24000)

print(f"🎉 Success! Audio saved to: {output_file}")

🔹 Target Folder: C:\Users\rajam\AppData\Local\tts\tts_models--multilingual--multi-dataset--xtts_v2
📦 Found 'model.pth': 1.74 GB
⚙️ Loading Configuration...
🧠 Initializing Model...
🏋️ Loading Weights (This may take 10s)...


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:/Users/rajam/AppData/Local/tts/tts_models--multilingual--multi-dataset--xtts_v2/model.pth/model.pth'

In [5]:
# FILE: 08_Voice_Test_Ultimate.ipynb

import os
import torch
import torchaudio
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

# --- 1. SETUP PATHS ---
base_path = os.path.join(os.getenv("LOCALAPPDATA"), "tts", "tts_models--multilingual--multi-dataset--xtts_v2")
checkpoint_path = os.path.join(base_path, "model.pth")
config_path = os.path.join(base_path, "config.json")
vocab_path = os.path.join(base_path, "vocab.json")
speaker_path = os.path.join(base_path, "speakers_xtts.pth")
my_voice_file = "my_voice.wav"

print(f"🔹 Target File: {checkpoint_path}")

# --- 2. RAW LOAD (The Bypass) ---
print("⚙️ Loading Config...")
config = XttsConfig()
config.load_json(config_path)

print("🧠 Initializing Empty Model...")
model = Xtts.init_from_config(config)

print("💉 Injecting Weights manually (Direct PyTorch Load)...")
# We load the dictionary directly, bypassing the confused library helper
state_dict = torch.load(checkpoint_path, map_location=torch.device("cpu"))

# XTTS checkpoints sometimes have the weights inside a "model" key
if "model" in state_dict:
    state_dict = state_dict["model"]

# Remove keys that might cause conflicts (training helpers)
keys_to_ignore = ["dvae", "torch_mel_spectrogram_style_encoder"]
cleaned_state_dict = {k: v for k, v in state_dict.items() if not any(x in k for x in keys_to_ignore)}

# Load the weights into our model
model.load_state_dict(cleaned_state_dict, strict=False)
model.cuda() # Move to GPU

print("✅ Model is ALIVE!")

# --- 3. GENERATE AUDIO ---
print("🎙️ Cloning your voice...")

# Load tokenizer manually since we skipped the helper
from TTS.tts.layers.xtts.tokenizer import VoiceBpeTokenizer
model.tokenizer = VoiceBpeTokenizer(vocab_file=vocab_path)

# Check for voice file
voice_input = my_voice_file if os.path.exists(my_voice_file) else speaker_path
if voice_input == speaker_path: print("⚠️ Using default voice (file not found)")

# Get conditioning
gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(audio_path=[voice_input])

# Speak
out = model.inference(
    text="I have finally broken free. I am speaking with your voice.",
    language="en",
    gpt_cond_latent=gpt_cond_latent,
    speaker_embedding=speaker_embedding,
    temperature=0.7,
)

# Save
torchaudio.save("output_final.wav", torch.tensor(out["wav"]).unsqueeze(0), 24000)
print("🎉 SUCCESS! Listen to 'output_final.wav'")

🔹 Target File: C:\Users\rajam\AppData\Local\tts\tts_models--multilingual--multi-dataset--xtts_v2\model.pth
⚙️ Loading Config...
🧠 Initializing Empty Model...
💉 Injecting Weights manually (Direct PyTorch Load)...


C:\Users\rajam\AppData\Local\Temp\ipykernel_30416\2956022955.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(checkpoint_path, map_location=torch

✅ Model is ALIVE!
🎙️ Cloning your voice...


AttributeError: 'GPT' object has no attribute 'gpt_inference'

In [7]:
# FILE: 08_Voice_Test_Final_Fix.ipynb

import os
import torch
import torchaudio
import requests
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
from TTS.tts.layers.xtts.tokenizer import VoiceBpeTokenizer

# --- 1. SETUP PATHS ---
base_path = os.path.join(os.getenv("LOCALAPPDATA"), "tts", "tts_models--multilingual--multi-dataset--xtts_v2")
checkpoint_path = os.path.join(base_path, "model.pth")
config_path = os.path.join(base_path, "config.json")
vocab_path = os.path.join(base_path, "vocab.json")
speaker_path = os.path.join(base_path, "speakers_xtts.pth")

print(f"🔹 Target File: {checkpoint_path}")

# --- 2. RAW LOAD ---
print("⚙️ Loading Config...")
config = XttsConfig()
config.load_json(config_path)

print("🧠 Initializing Model...")
model = Xtts.init_from_config(config)

print("💉 Injecting Weights manually...")
state_dict = torch.load(checkpoint_path, map_location=torch.device("cpu"))

# Handle nested keys if present
if "model" in state_dict:
    state_dict = state_dict["model"]

# Clean incompatible keys
keys_to_ignore = ["dvae", "torch_mel_spectrogram_style_encoder"]
cleaned_state_dict = {k: v for k, v in state_dict.items() if not any(x in k for x in keys_to_ignore)}

model.load_state_dict(cleaned_state_dict, strict=False)
model.cuda()

# --- 3. THE FIX: INITIALIZE INFERENCE ENGINE ---
print("🔧 Connecting Inference Engine (The missing step)...")
model.gpt.init_gpt_for_inference(kv_cache=True)  # <--- THIS IS THE MAGIC LINE

print("✅ Model is FULLY ALIVE!")

# --- 4. GENERATE AUDIO ---
print("🎙️ Cloning voice...")

# Setup Tokenizer
model.tokenizer = VoiceBpeTokenizer(vocab_file=vocab_path)

# Download a known working sample to ensure input is perfect
test_voice_url = "https://github.com/coqui-ai/TTS/raw/dev/TTS/utils/assets/baker_opening.wav"
my_voice_file = "my_voice.wav"

if not os.path.exists(my_voice_file):
    print("⬇️ Downloading test voice...")
    response = requests.get(test_voice_url)
    with open(my_voice_file, "wb") as f:
        f.write(response.content)

# Get conditioning
gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(audio_path=[my_voice_file])

# Speak
out = model.inference(
    text="Success! I am speaking. The code is finally working properly.",
    language="en",
    gpt_cond_latent=gpt_cond_latent,
    speaker_embedding=speaker_embedding,
    temperature=0.7,
)

# Save
output_path = "output_success.wav"
torchaudio.save(output_path, torch.tensor(out["wav"]).unsqueeze(0), 24000)
print(f"🎉 FINAL SUCCESS! Listen to '{output_path}'")

🔹 Target File: C:\Users\rajam\AppData\Local\tts\tts_models--multilingual--multi-dataset--xtts_v2\model.pth
⚙️ Loading Config...
🧠 Initializing Model...
💉 Injecting Weights manually...


C:\Users\rajam\AppData\Local\Temp\ipykernel_30416\4143121508.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(checkpoint_path, map_location=torch

🔧 Connecting Inference Engine (The missing step)...
✅ Model is FULLY ALIVE!
🎙️ Cloning voice...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token.As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


🎉 FINAL SUCCESS! Listen to 'output_success.wav'
